In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import glob
from datetime import datetime
import os

#
# AROC CALCULATION BY FORECAST DAY - NaN-SAFE VERSION
# ----------------------------------------------------
#
# Properly handles NaN values in probability columns
#
########################################

# ===== CONFIGURATION =====
#DATA_PATH = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/DATA_preVerif/2024_2025_49r1/"
DATA_PATH = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/DATA_preVerif/2020_2021/"
OUTPUT_PATH = "./verification_results/"

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

PRECIP_TYPES = ['rain', 'snow', 'sleet', 'freezrain', 'freezdrizzle', 'icepellets']

COLORS = {
    'rain': '#00CC00',        # Bright green (from your plot)
    'snow': '#0000FF',        # Blue (from your plot)
    'sleet': '#00CCCC',       # Cyan/turquoise (from your plot)
    'freezrain': '#FF0000',   # Red (from your plot)
    'freezdrizzle': '#FF1493', # Deep pink (as requested)
    'icepellets': '#FF9900'   # Orange (from your plot)
}

NICE_NAMES = {
    'rain': 'Rain',
    'snow': 'Snow',
    'sleet': 'Sleet',
    'freezrain': 'Freezing Rain',
    'freezdrizzle': 'Freezing Drizzle',
    'icepellets': 'Ice Pellets'
}

print("="*70)
print("AROC CALCULATION BY FORECAST DAY")
print("="*70)
print(f"Data path: {DATA_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print()

# ===== LOAD DATA =====
print("1. Loading data...")
start_time = datetime.now()

files = sorted(glob.glob(DATA_PATH + "matched_pairs_*.parquet"))
print(f"   Found {len(files)} files")

if len(files) == 0:
    print("ERROR: No parquet files found!")
    exit(1)

df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

elapsed = (datetime.now() - start_time).total_seconds()
print(f"   Loaded {len(df):,} matched pairs in {elapsed:.1f} seconds")
print(f"   Date range: {df['forecast_date'].min()} to {df['forecast_date'].max()}")
print(f"   Forecast steps: {df['forecast_step'].min()}h to {df['forecast_step'].max()}h")
print()

# ===== DEFINE DAY GROUPINGS =====
day_groups = {
    1: [0, 3, 6, 9, 12, 15, 18, 21],
    2: [24, 27, 30, 33, 36, 39, 42, 45],
    3: [48, 51, 54, 57, 60, 63, 66, 69],
    4: [72, 75, 78, 81, 84, 87, 90, 93],
    5: [96, 99, 102, 105, 108, 111, 114, 117],
    6: [120, 123, 126, 129, 132, 135, 138, 141],
    7: [144, 150, 156, 162, 168]
}

days = sorted(day_groups.keys())
print(f"2. Calculating AROC by forecast day...")
print()

# Store results
aroc_results = {ptype: [] for ptype in PRECIP_TYPES}
n_samples = {ptype: [] for ptype in PRECIP_TYPES}
n_events = {ptype: [] for ptype in PRECIP_TYPES}

# Calculate AROC for each day
for day_idx, day in enumerate(days, start=1):
    
    steps_in_day = day_groups[day]
    step_range = f"{steps_in_day[0]}-{steps_in_day[-1]}h"
    
    print(f"   [{day_idx:2d}/{len(days)}] Day {day} ({step_range})", end='', flush=True)
    
    # Filter data for all steps in this day
    df_day = df[df['forecast_step'].isin(steps_in_day)]
    n_total = len(df_day)
    
    print(f" | {n_total:,} samples", end='', flush=True)
    
    results_day = []
    
    for ptype in PRECIP_TYPES:
        
        obs_col = f'obs_{ptype}'
        prob_col = f'prob_{ptype}'
        
        # Get observations and probabilities
        obs = df_day[obs_col].values
        prob = df_day[prob_col].values / 100.0  # Convert % to [0,1]
        
        # ===== KEY FIX: Remove NaN values =====
        # Create mask for valid (non-NaN) probabilities
        valid_mask = ~np.isnan(prob)
        obs_valid = obs[valid_mask]
        prob_valid = prob[valid_mask]
        
        # Count events (after removing NaN)
        n_event = obs_valid.sum()
        n_valid = len(obs_valid)
        n_samples[ptype].append(n_valid)
        n_events[ptype].append(n_event)
        
        # Calculate AROC if we have both events and non-events
        if n_event > 0 and n_event < n_valid:
            try:
                aroc = roc_auc_score(obs_valid, prob_valid)
                aroc_results[ptype].append(aroc)
                results_day.append(f"{ptype[:4]}={aroc:.3f}")
            except Exception as e:
                aroc_results[ptype].append(np.nan)
                results_day.append(f"{ptype[:4]}=ERR")
        else:
            # If all events or no events, AROC is undefined
            aroc_results[ptype].append(np.nan)
            if n_event == 0:
                results_day.append(f"{ptype[:4]}=NoEvt")
            else:
                results_day.append(f"{ptype[:4]}=AllEvt")
    
    print(f" | {', '.join(results_day[:3])}")

print()

# ===== SUMMARY STATISTICS =====
print("3. Summary statistics:")
print()
print(f"{'Type':<20} {'Mean AROC':<12} {'Min AROC':<12} {'Max AROC':<12} {'Valid Days':<12}")
print("-"*70)

for ptype in PRECIP_TYPES:
    aroc_vals = np.array(aroc_results[ptype])
    valid_aroc = aroc_vals[~np.isnan(aroc_vals)]
    
    if len(valid_aroc) > 0:
        mean_aroc = valid_aroc.mean()
        min_aroc = valid_aroc.min()
        max_aroc = valid_aroc.max()
        n_valid = len(valid_aroc)
    else:
        mean_aroc = np.nan
        min_aroc = np.nan
        max_aroc = np.nan
        n_valid = 0
    
    print(f"{NICE_NAMES[ptype]:<20} {mean_aroc:>10.3f}  {min_aroc:>10.3f}  {max_aroc:>10.3f}  {n_valid:>5d}/{len(days)}")

print()

# ===== CREATE PLOT =====
print("4. Creating plot...")

fig, ax = plt.subplots(figsize=(12, 8))

# Plot each precipitation type
for ptype in PRECIP_TYPES:
    aroc_vals = np.array(aroc_results[ptype])
    
    # Count valid (non-NaN) values
    n_valid = np.sum(~np.isnan(aroc_vals))
    
    if n_valid > 0:
        # Plot line
        ax.plot(days, aroc_vals, 
                marker='o', 
                linewidth=2.5, 
                markersize=8,
                label=f"{NICE_NAMES[ptype]}",
                color=COLORS[ptype],
                alpha=0.8)

# Reference line at AROC = 0.5 (random skill)
ax.axhline(y=0.5, color='black', linestyle='--', linewidth=1.5, 
           label='No skill (0.5)', alpha=0.5)

# Formatting
ax.set_xlabel('Forecast Day', fontsize=14, fontweight='bold')
ax.set_ylabel('Area Under ROC Curve (AROC)', fontsize=14, fontweight='bold')
ax.set_title('Forecast Skill by Forecast Day - All Precipitation Types', 
             fontsize=16, fontweight='bold', pad=20)

# Set y-axis limits
ax.set_ylim([0.45, 1.0])
ax.set_xlim([0.5, len(days) + 0.5])

# Set x-axis ticks to show days
ax.set_xticks(days)
ax.set_xticklabels([f'D{d}' for d in days])

# Grid
ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
ax.set_axisbelow(True)

# Legend
ax.legend(loc='best', fontsize=11, framealpha=0.95, 
          edgecolor='black', fancybox=False)

# Tick formatting
ax.tick_params(axis='both', labelsize=11)

# Add text box with dataset info
textstr = f'Dataset: {len(df):,} matched pairs\n'
textstr += f'Date range: {df["forecast_date"].min()}-{df["forecast_date"].max()}\n'
textstr += f'Forecast days: {len(days)} days'

props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

plt.tight_layout()
plt.savefig(OUTPUT_PATH + 'aroc_by_day.png', dpi=300, bbox_inches='tight')
print(f"   Saved: {OUTPUT_PATH}aroc_by_day.png")
print()

# ===== SAVE RESULTS TO CSV =====
print("5. Saving results to CSV...")

# Create DataFrame with results
results_df = pd.DataFrame({
    'forecast_day': days,
    'step_range': [f"{day_groups[d][0]}-{day_groups[d][-1]}h" for d in days]
})

for ptype in PRECIP_TYPES:
    results_df[f'aroc_{ptype}'] = aroc_results[ptype]
    results_df[f'n_events_{ptype}'] = n_events[ptype]
    results_df[f'n_samples_{ptype}'] = n_samples[ptype]

results_df.to_csv(OUTPUT_PATH + 'aroc_by_day.csv', index=False)
print(f"   Saved: {OUTPUT_PATH}aroc_by_day.csv")
print()

# ===== CREATE DETAILED PLOT FOR EACH TYPE =====
print("6. Creating individual plots for each precipitation type...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, ptype in enumerate(PRECIP_TYPES):
    ax = axes[idx]
    
    aroc_vals = np.array(aroc_results[ptype])
    events = np.array(n_events[ptype])
    
    # Plot AROC
    ax.plot(days, aroc_vals, 
            marker='o', 
            linewidth=2.5, 
            markersize=8,
            color=COLORS[ptype],
            label='AROC')
    
    # Reference line
    ax.axhline(y=0.5, color='black', linestyle='--', linewidth=1, alpha=0.5)
    
    # Twin axis for event frequency
    ax2 = ax.twinx()
    ax2.bar(days, events, alpha=0.3, color=COLORS[ptype], width=0.4, label='Events')
    ax2.set_ylabel('Number of Events', fontsize=10, color=COLORS[ptype])
    ax2.tick_params(axis='y', labelcolor=COLORS[ptype])
    
    # Formatting
    ax.set_xlabel('Forecast Day', fontsize=11)
    ax.set_ylabel('AROC', fontsize=11, fontweight='bold')
    ax.set_title(NICE_NAMES[ptype], fontsize=13, fontweight='bold')
    ax.set_ylim([0.45, 1.0])
    ax.set_xlim([0.5, len(days) + 0.5])
    ax.set_xticks(days)
    ax.set_xticklabels([f'D{d}' for d in days])
    ax.grid(True, alpha=0.3)
    
    # Calculate mean AROC
    valid_aroc = aroc_vals[~np.isnan(aroc_vals)]
    if len(valid_aroc) > 0:
        mean_aroc = valid_aroc.mean()
        ax.text(0.05, 0.95, f'Mean AROC: {mean_aroc:.3f}', 
                transform=ax.transAxes, fontsize=10,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(OUTPUT_PATH + 'aroc_by_day_individual.png', dpi=300, bbox_inches='tight')
print(f"   Saved: {OUTPUT_PATH}aroc_by_day_individual.png")
print()

# ===== FINAL SUMMARY =====
total_elapsed = (datetime.now() - start_time).total_seconds()

print("="*70)
print("ANALYSIS COMPLETE!")
print("="*70)
print(f"Total time: {total_elapsed:.1f} seconds")
print()
print("Output files:")
print(f"  - {OUTPUT_PATH}aroc_by_day.png (combined plot)")
print(f"  - {OUTPUT_PATH}aroc_by_day_individual.png (individual plots)")
print(f"  - {OUTPUT_PATH}aroc_by_day.csv (data table)")
print()

# Key findings
print("Key findings:")
print()
for ptype in PRECIP_TYPES:
    aroc_vals = np.array(aroc_results[ptype])
    valid_aroc = aroc_vals[~np.isnan(aroc_vals)]
    
    if len(valid_aroc) > 0:
        mean_aroc = valid_aroc.mean()
        
        # Classify skill level
        if mean_aroc >= 0.9:
            skill_level = "Excellent"
        elif mean_aroc >= 0.8:
            skill_level = "Good"
        elif mean_aroc >= 0.7:
            skill_level = "Fair"
        elif mean_aroc >= 0.6:
            skill_level = "Poor"
        else:
            skill_level = "Very Poor"
        
        print(f"  {NICE_NAMES[ptype]:<20}: Mean AROC = {mean_aroc:.3f} ({skill_level})")

print()
print("Interpretation:")
print("  AROC = 1.0  : Perfect forecast")
print("  AROC = 0.9+ : Excellent skill")
print("  AROC = 0.8+ : Good skill")
print("  AROC = 0.7+ : Fair skill")
print("  AROC = 0.5  : No skill (random)")

AROC CALCULATION BY FORECAST DAY
Data path: /ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/DATA_preVerif/2020_2021/
Output path: ./verification_results/

1. Loading data...
   Found 177 files
   Loaded 10,868,520 matched pairs in 1.5 seconds
   Date range: 20201016 to 20210410
   Forecast steps: 0h to 168h

2. Calculating AROC by forecast day...

   [ 1/7] Day 1 (0-21h) | 1,637,841 samples

KeyError: 'obs_freezdrizzle'

In [2]:
import pandas as pd
import glob

DATA_PATH = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/DATA_preVerif/2024_2025_49r1/"

print("Investigating NaN Probabilities")
print("="*70)

# Load first file
files = sorted(glob.glob(DATA_PATH + "matched_pairs_*.parquet"))
df = pd.read_parquet(files[0])

print(f"First file: {files[0]}")
print(f"Rows: {len(df):,}")
print()

# Show columns
print("Columns in file:")
for col in df.columns:
    print(f"  {col}")
print()

# Check first 10 rows
print("First 10 rows of key columns:")
print(df[['forecast_step', 'obs_rain', 'prob_rain', 'obs_snow', 'prob_snow']].head(10))
print()

# Check NaN counts
print("NaN counts per column:")
for col in df.columns:
    n_nan = df[col].isna().sum()
    pct_nan = 100 * n_nan / len(df)
    print(f"  {col:25s}: {n_nan:8,} NaN ({pct_nan:5.1f}%)")
print()

# Check if ALL probabilities are NaN
all_prob_cols = [c for c in df.columns if c.startswith('prob_')]
print(f"Probability columns: {all_prob_cols}")
print()

for col in all_prob_cols:
    n_nan = df[col].isna().sum()
    n_valid = len(df) - n_nan
    if n_valid > 0:
        print(f"{col}: {n_valid} valid values, {n_nan} NaN")
        print(f"  Sample valid values: {df[col].dropna().head(10).tolist()}")
    else:
        print(f"{col}: ALL NaN!")
print()

# Check the actual data types
print("Data types:")
print(df.dtypes)

Investigating NaN Probabilities
First file: /ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/DATA_preVerif/2024_2025_49r1/matched_pairs_20241112.parquet
Rows: 74,246

Columns in file:
  forecast_date
  valid_date
  forecast_step
  valid_hour
  latitude
  longitude
  elevation
  synop_code
  obs_rain
  obs_snow
  obs_wetsnow
  obs_sleet
  obs_freezrain
  obs_freezdrizzle
  obs_icepellets
  prob_rain
  prob_snow
  prob_wetsnow
  prob_sleet
  prob_freezrain
  prob_freezdrizzle
  prob_icepellets

First 10 rows of key columns:
   forecast_step  obs_rain  prob_rain  obs_snow  prob_snow
0              0         0  35.294113         0   0.000000
1              0         0   0.000000         0   0.000000
2              0         1   7.843140         0   0.000000
3              0         0   0.000000         1   0.000000
4              0         0   0.000000         1   0.000000
5              0         0   3.921570         0   0.000000
6              0         0   0.000000         0   1.960793
7